In [ ]:
import numpy as np

from src import coupling_error_convergence_funcs as study
from src.lib.reader import  reader as params
from src.lib.model import setup_LIB as setup 

Input basic simulation and physical parameters from input file and a solution of the algebraic system corresponding to initial condition of differential system as initial guess is obtained.

In [ ]:
inputFile = 'src/input/'           
phy_options, sim_options = params.readInputs(inputFile)

options_electrolyte, options_cathode = setup.getSetup(phy_options, sim_options, invertBV=False)

Ui = np.r_[options_electrolyte['y0'], options_cathode['y0']] # Initial condition solution

Setup of the convergence study

In [ ]:
cvType = 'constant' # Constant Voltage (CV) mode simulation

order_vec= np.array(range(1, 4 + 1)) # Vector of orders of coupling studied

nt_vec = np.array([ 11,  23,  50,  70,  97, 135, 188, 260, 361, 501], dtype=int) # Vector of the numbers of coupling intervals (nt_vec) to reproduce the study in paper

# nt_vec = np.array([11, 30, 50], dtype=int) # nt_vec for test

tini_dim = 0.0 # Time in s at which the initial CC mode simulation starts
tccE_dim = 11.0 # Time in s at which the initial CC mode simulation ends and CV mode simulation starts
tend_dim = 101.0 # Time in s at which CV mode simulation ends

# Dimensionless times
tc = options_electrolyte['parameters']['t_c'] # Timescale to enable a dimensionless simulation

tini = tini_dim/tc # Dimensionless time for CC mode start
t_CC_end = tccE_dim/tc # Dimensionless time for CC mode end or CV mode simulation start
tend = tend_dim/tc # Dimensionless time for CV mode simulation end 

t_md_start = t_CC_end # Dimensionless time for the start of multi-domain coupled simulations
t_md_end = tend # Dimensionless time for the start of multi-domain coupled simulations

Initial CC simulation until t=11s. Similar to practical scenario, we charge the battery in CC mode before moving to CV mode where multi-domain integration method is studied.

In [ ]:
options_cathode['sim_type']="monolithic"
options_electrolyte['sim_type']="monolithic"

out_cc = study.initial_CC_sim(
                        tini,
                        t_CC_end,
                        y0_cc=Ui,
                        rtol=1e-9,
                        options_electrolyte=options_electrolyte,
                        options_cathode=options_cathode
)


Setting up CV mode LIB simulations

In [ ]:
Umean = out_cc.y[-1, -1]*options_electrolyte['parameters']['phi_c'] # in Volts
Ucell_t = lambda t: Umean 

options_cathode['parameters']['ChargeType'] = "CV"
options_cathode['parameters']['phi_s_L'] = Ucell_t

Quasi-exact reference monolithic solution

In [ ]:
options_cathode['sim_type']="monolithic"
options_electrolyte['sim_type']="monolithic"

ref_sol_monolithic = study.get_ref_sol(
                      y0=out_cc.y[:,-1],
                      tstart=t_md_start,
                      tend=t_md_end,
                      rtol=1e-12,
                      options_electrolyte=options_electrolyte,
                      options_cathode=options_cathode
)

Voltage evolution of the setup

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}' 
rcParams['text.usetex'] = True

time_vec = ref_sol_monolithic.t[:] * tc
Ucell_vec = ref_sol_monolithic.y[-1, :] * options_electrolyte['parameters']['phi_c']

plt.figure()


plt.plot(out_cc.t[:-1] * tc,
         out_cc.y[-1, :-1] * options_electrolyte['parameters']['phi_c'],
         color='k', lw=2, ls='--',
         label=r"$\mathrm{Initial\ monolithic\ simulation\ in\ CC\ mode}$")

plt.plot(time_vec, Ucell_vec,
         color='C0', lw=2, ls='-',
         label=r"$\mathrm{Coupled\ multi\text{-}domain\ simulation\ in\ CV\ mode}$")
         
plt.xlabel(r"$t\ \mathrm{(s)}$", fontsize=15)
plt.ylabel(r"$U_{cell}\ \mathrm{(V)}$", fontsize=15)
plt.grid(ls=':')
plt.legend(framealpha=.75, fancybox=True,
           loc='lower right', numpoints=1, fontsize=10)
plt.tight_layout()

Multi-domain LIB simulation convergence study

In [ ]:
options_cathode['sim_type']="md_coupling_vars"
options_electrolyte['sim_type']="md_coupling_vars"

nparallel=8

# explicit coupling
sols_md_explicit = study.convergence_study_loop(
                                        nt_vec,
                                        order_vec,
                                        t_md_start,
                                        t_md_end,
                                        y0_global=out_cc.y[:,-1],
                                        bExplicitCoupling=True,
                                        options_electrolyte=options_electrolyte,
                                        options_cathode=options_cathode,
                                        outRef=ref_sol_monolithic,
                                        bMDSim_v1=False,
                                        nparallel=nparallel
)
    
# implicit coupling
sols_md_implicit = study.convergence_study_loop(
                                        nt_vec,
                                        order_vec,
                                        t_md_start,
                                        t_md_end,
                                        y0_global=out_cc.y[:,-1],
                                        bExplicitCoupling=False,
                                        options_electrolyte=options_electrolyte,
                                        options_cathode=options_cathode,
                                        outRef=ref_sol_monolithic,
                                        bMDSim_v1=False,
                                        nparallel=nparallel
)

Error evaluation

In [ ]:
error_explicit = study.get_errors(
                    nt_vec,
                    order_vec,
                    sols_md_explicit,
                    ref_sol_monolithic
)

error_implicit = study.get_errors(
                    nt_vec,
                    order_vec,
                    sols_md_implicit,
                    ref_sol_monolithic
)

Error Plots

In [ ]:
dt_vec = abs(t_md_end - t_md_start)/(nt_vec - 1)

# Skipping index to plot theorectical order curves
qsList_exp = {1:1,2:2,3:1,4:2}
qsList_imp = {1:1,2:2,3:1,4:1}

nt_plt_vecE, dt_plt_vecE, err_plt_vecE, nt_order_vecE, dt_order_vecE, th_curve_vecE = study.get_plt_curves(
    nt_vec,
    dt_vec,
    order_vec,
    error_explicit,
    qsList=qsList_exp
)

nt_plt_vecI, dt_plt_vecI, err_plt_vecI, nt_order_vecI, dt_order_vecI, th_curve_vecI = study.get_plt_curves(
    nt_vec,
    dt_vec,
    order_vec,
    error_implicit,
    qsList=qsList_imp
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}' 
rcParams['text.usetex'] = True
plt.style.use = 'science'

clrmap = plt.get_cmap("tab10") 
clr =  [clrmap(i) for i in range(10)]
malpha = 0.75
fzA=15

mfc = [(i[0], i[1], i[2], malpha) for i in clr]

mkr = ['o', 'v', 's', 'd', 'V', '^' ]
# mkrI = ['*', 'v', 's', 'd', 'V', '^' ]

bDeltatc = False # Including error vs coupling intervals plot
fig1, ax1 = plt.subplots()
if bDeltatc:
    fig2, ax2 = plt.subplots()

for k, current_order in enumerate(order_vec[:]):
    ax1.loglog(nt_plt_vecE[k], err_plt_vecE[k], 
               color=clr[k], marker=mkr[k], linestyle='-', ms=8, markeredgewidth=1, mfc='none')
    
    ax1.loglog(nt_order_vecE[k], th_curve_vecE[k],
                color='k', linestyle='--', marker=None, lw=1, alpha=0.5)
    
    if bDeltatc:
        ax2.loglog(dt_plt_vecE[k]*tc, err_plt_vecE[k], 
                color=clr[k], marker=mkr[k], linestyle='-', ms=8, markeredgewidth=1, mfc='none')
        
        ax2.loglog(dt_order_vecE[k]*tc, th_curve_vecE[k],
                    color='k', linestyle='--', marker=None, lw=1, alpha=0.5)
    
for k, current_order in enumerate(order_vec[:]):
    ax1.loglog(nt_plt_vecI[k], err_plt_vecI[k], 
               color=clr[k], marker=mkr[k], linestyle='-', ms=7, markeredgewidth=1, mfc=mfc[k])
    
    ax1.loglog(nt_order_vecI[k], th_curve_vecI[k],
                color='k', linestyle='--', marker=None, lw=1, alpha=0.5)
    
    ax1.loglog(np.nan, np.nan, 
                color=clr[k], marker=mkr[k], linestyle='-', 
                markersize=7, markeredgewidth=1, mfc=mfc[k], 
                label=f"$p={current_order-1}$")
    
    if bDeltatc:
        ax2.loglog(dt_plt_vecI[k]*tc, err_plt_vecI[k], 
                color=clr[k], marker=mkr[k], linestyle='-', ms=7, markeredgewidth=1, mfc=mfc[k])
        
        
        ax2.loglog(dt_order_vecI[k]*tc, th_curve_vecI[k],
                    color='k', linestyle='--', marker=None, lw=1, alpha=0.5)
        
        ax2.loglog(np.nan, np.nan, 
                    color=clr[k], marker=mkr[k], linestyle='-', 
                    markersize=7, markeredgewidth=1, mfc=mfc[k], 
                    label=f"$p={current_order-1}$")

ax1.grid(ls=':')
ax1.legend(loc='lower left', framealpha=0.75, ncol=1, numpoints=1, fontsize=fzA-4)
ax1.set_xlabel(r"$\mathcal{N}_t$", fontsize=fzA)
ax1.set_ylabel(r"$\mathrm{error}$", fontsize=fzA)
ax1.set_xlim(10**(0.5*np.log10(np.min(nt_vec))), 10**(1.1*np.log10(np.max(nt_vec))))
fig1.tight_layout()

if bDeltatc:
    ax2.grid(ls=':')
    ax2.legend(loc='upper left', framealpha=0.75, ncol=1, numpoints=1, fontsize=fzA-4)
    ax2.set_xlabel(r"$\Delta t_c$ (s)", fontsize=fzA)
    ax2.set_ylabel(r"$\mathrm{error}$", fontsize=fzA)
    ax2.set_xlim(10**(1.25*np.log10(np.min(dt_vec*tc))), 10**(0.75*np.log10(np.max(dt_vec*tc))))
    fig2.tight_layout()

Save results 

In [ ]:
bSavefig = False
if bSavefig:
    savepath = 'output/'
    savefile = savepath+f"fig_{cvType}CV_both_tfin{int(tend_dim)}_pmax{order_vec[-1]}"
    fig1.savefig(savefile+".pdf", dpi=600)
    if bDeltatc:
        fig2.savefig(savefile+"_delta_tc"+".pdf", dpi=600)